# Bonds: Find Necessary Fields

Often more fields are required than are available by default. One may want to see the ratings for the bond issuer. Some fields name in API are not self-explanatory. Also some the returned values are often encoded and not readable by human.

Unfortunately, LSEG does not provide a list of available fields and their explanation. There are a few things that can be done.

In [ ]:
import lseg.data as ld
from lseg.data.content import search
import os
import pandas as pd

os.environ["LD_LIB_CONFIG_PATH"] = "../../Configuration"
ld.open_session()

The scope is government and corporate bonds:

In [ ]:
view = search.Views.GOV_CORP_INSTRUMENTS

Let us get a list of _corporate_ bonds, both _Active_ and _Inactive_, issued on a _specific_ date. This can be done through filters.

In [ ]:
date_str = '2026-03-11'
filter_str = f"(DbType eq 'CORP' and RCSAssetCategory eq 'A:J' and ((IssueDate ge {date_str} and IssueDate le {date_str})))"

When the required fields are unknown, one can pull all fields for a small subset of bonds and apply own common sense to select the required.

In [ ]:
metadata_response = search.metadata.Definition(view=view).get_data()
all_fields = metadata_response.data.df.reset_index()["level_0"].to_list()

Pull all fields the data from LSEG API

In [ ]:
response = search.Definition(
        view=view,
        top=100,
        filter=filter_str,
        select = ','.join(all_fields)
    ).get_data()
df = response.data.df
df

One can do preselection of fields to speed up the process

In [ ]:
rating_fields = [x for x in all_fields if 'Rating' in x]
response = search.Definition(
        view=view,
        top=100,
        filter=filter_str,
        select = ','.join(rating_fields)
    ).get_data()
df = response.data.df
df

After the selection process the API request can look like below.

In [ ]:
view = search.Views.GOV_CORP_INSTRUMENTS
date_str = '2026-03-11'
filter_list = ["DbType eq 'CORP'",  # Corporate bonds
               "RCSAssetCategory eq 'A:J'",  # Include only bonds, and not CP or CD
               "IsConvertible eq false",  # Exclude convertable bonds
               "RCSCouponTypeGenealogy eq 'M:1EU\\A:C1\\A:25'",  # Select Coupon Type as Plain Vanilla Fixed Coupon
               "RCSCurrency in ('C:6' 'C:5' 'C:3' 'C:4')",  # Select EUR, USD, GBP, JPY
               f"(IssueDate ge {date_str} and IssueDate le {date_str})"  # Define Issue Date range
               ]
filter_str = '(' + ' and '.join(filter_list) + ')'
select_list = ['DTSubjectName',
             'DBSTicker',
             'MoodyIssuerRating',
             'FitchIssuerRating',
             'IssueRating',
             'IssuerRatingLatest',
             'CouponRate',
             'MaturityDate',
             'IssueDate',
             'ISIN',
             'RIC',
             'RCSCouponTypeGenealogy',
             'DbType',
             'IsActive',
             'RCSRiskOrganisationCountry',
             'RCSCurrency',
             'RCSCountryGenealogy',
             'EJVAssetID',
             'BusinessEntity',
             'PI',
             'RCSCurrencyLeaf',
             'RCSCountryLeaf',
             'DbTypeDescription',
             'InstrumentTypeDescription',
             'FaceIssuedUSD']
select_str = ','.join(select_list)
response = search.Definition(
        view=view,
        top=10000,
        filter=filter_str,
        select=select_str
    ).get_data()
df = response.data.df
df

It remains a problem to get the rating of the bonds or the issuers. Some rating fields are not always returned by API.

One may also consider to combine ratings if they are missing. For example, a waterfall approach can be applied where first Moody rating is chosen, then Fitch if Moody is missing and so on.

In [ ]:
df['Rating'] = df['MoodyIssuerRating']
df.loc[df['Rating'].isna(), 'Rating'] =  df.loc[df['Rating'].isna(), 'FitchIssuerRating']
df.loc[df['Rating'].isna(), 'Rating'] =  df.loc[df['Rating'].isna(), 'IssuerRatingLatest']
df

In [ ]:
ld.close_session()